# Step 05 — Resume Parser fine-tuning on Kaggle (one T4)

Use a fresh **GPU T4 x2** session with **Internet enabled**; this notebook exposes only GPU 0 to training. Enable the named secrets for this notebook under Add-ons → Secrets before **Save Version → Save & Run All**. W&B tracking and post-evaluation are enabled by default; the final adapter publishes to `md-zain/resume-parser` on `main`. No interactive login or upload prompts are used.

The default is 4-bit Unsloth Qwen2.5-3B, FP16, 1,024 tokens, batch 1, accumulation 16, paged 8-bit AdamW, and LoRA rank/alpha 16. Whole examples that exceed the context limit are excluded and reported. Training and inference share the compact prompt; old schema-prompt evaluation scores are not a matched baseline.

The first code cell sets GPU visibility and checks CUDA in a **child process**, so the notebook can install dependencies before importing Unsloth or Torch in its own kernel. Dependency pins are checked against published releases, and the installed Torch/CUDA stack is constrained to its existing versions.

Source, secrets, `.env`, dataset files, and download caches stay under `/tmp`. Only checkpoints, adapter files, and anonymous evaluation tables go under `/kaggle/working`. Detailed evaluation records containing resume text remain under the private `/tmp` checkout. Hugging Face commits provide recovery across fresh Kaggle sessions. Do not publish sensitive evaluation outputs.

Training is capped at 100 optimizer steps and requests a final checkpoint at a step boundary after a 150-minute session budget. Setup, compilation, uploads, and slow steps can exceed estimates; this is not a hard three-hour guarantee or a change to Kaggle's quota.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Cell 1 — Select one GPU before any Torch/Unsloth import; check the current accelerator.
import subprocess
import sys
import time
from pathlib import Path

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook in Kaggle with a GPU accelerator enabled.')
if any(name in sys.modules for name in ('torch', 'unsloth', 'transformers', 'peft', 'trl')):
    raise RuntimeError('Start a fresh Kaggle session and run from the first cell before importing the training stack.')
if int(os.getenv('WORLD_SIZE', '1')) != 1:
    raise RuntimeError('Use the normal single-process notebook kernel, not a distributed launcher.')
SESSION_STARTED_AT = time.monotonic()
PRIVATE_ROOT = Path('/tmp/resume-parser-kaggle')
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)
PRIVATE_ROOT.chmod(0o700)
os.environ['HF_HOME'] = str(PRIVATE_ROOT / 'huggingface')
os.environ['HF_TOKEN_PATH'] = str(PRIVATE_ROOT / 'huggingface/token')
os.environ['WANDB_DIR'] = str(PRIVATE_ROOT / 'wandb')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# nvidia-smi lists physical devices; the child-process check confirms CUDA visibility.
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-c', (
    'import torch; '
    'print("torch.cuda.device_count():", torch.cuda.device_count()); '
    'assert torch.cuda.is_available() and torch.cuda.device_count() == 1, '
    '"Expected exactly one visible CUDA GPU; start a fresh GPU session."; '
    'print("Visible GPU:", torch.cuda.get_device_name(0))'
)], check=True)


In [ ]:
# Cell 2 — Read enabled Kaggle Secrets; never print their values.
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
# [LIST OF ENV VARIABLE NAMES] — add optional project keys here.
ENV_VARIABLE_NAMES = [
    'HF_TOKEN', 'GITHUB_TOKEN', 'HF_DATASET_REVISION',
    'WANDB_API_KEY', 'WANDB_ENTITY', 'WANDB_PROJECT', 'TYPESAFE_API_KEY',
]
try:
    hf_token = s.get_secret('HF_TOKEN').strip()
except Exception:
    raise RuntimeError('Enable a write-capable HF_TOKEN in Kaggle Add-ons → Secrets for this notebook.') from None
if not hf_token:
    raise RuntimeError('HF_TOKEN is empty. Update the Kaggle secret before running.')
os.environ['HF_TOKEN'] = hf_token
loaded_secret_names = {'HF_TOKEN'}
for name in ENV_VARIABLE_NAMES:
    if name == 'HF_TOKEN':
        continue
    try:
        value = s.get_secret(name)
    except Exception:
        continue
    if value:
        os.environ[name] = value
        loaded_secret_names.add(name)
print('Enabled secret names:', ', '.join(sorted(loaded_secret_names)))


In [ ]:
# Cell 3 — Clone public/private GitHub source, or copy an attached Kaggle source folder.
import base64
import shutil
import socket
from urllib.parse import urlsplit

REPO_URL = 'https://github.com/Muhammad-Zain01/resume-parser-model.git'  # [REPO_URL]
SOURCE_DIR = None  # Optional: '/kaggle/input/YOUR-SOURCE-DATASET/resume-parser-model'
PROJECT_ROOT = PRIVATE_ROOT / 'source'
MARKER = Path('src/steps/05_fine_tuning/fine_tuning.py')

def check_dns(host):
    for attempt in range(3):
        try:
            socket.getaddrinfo(host, 443)
            return
        except socket.gaierror:
            if attempt < 2:
                time.sleep(2)
    raise RuntimeError(
        f'Cannot resolve {host}. Enable Internet in Kaggle notebook settings and save those '
        'settings before Save & Run All. If Internet is already enabled, restart the session '
        'and retry; this is a network/DNS problem. For GitHub-only outages, attach a source '
        'folder as Kaggle input and set SOURCE_DIR above. Pip and Hugging Face still need Internet.'
    )

if SOURCE_DIR:
    source = Path(SOURCE_DIR).resolve()
    if not (source / MARKER).is_file():
        raise FileNotFoundError('SOURCE_DIR must contain src/steps/05_fine_tuning/fine_tuning.py.')
    if PROJECT_ROOT.exists():
        raise RuntimeError('Source is already staged. Start a fresh session before changing SOURCE_DIR.')
    shutil.copytree(source, PROJECT_ROOT, ignore=shutil.ignore_patterns(
        '.git', '.env', '.env.*', '.data', '.venv', '__pycache__', '*.ipynb_checkpoints',
    ))
else:
    parsed = urlsplit(REPO_URL)
    if (parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username
            or parsed.password or parsed.query or parsed.fragment or parsed.port):
        raise ValueError('Use a clean https://github.com/OWNER/REPO.git URL without credentials or query parameters.')
    check_dns('github.com')
    git_env = os.environ.copy()
    git_env['GIT_TERMINAL_PROMPT'] = '0'
    github_token = os.getenv('GITHUB_TOKEN', '').strip()
    encoded_auth = ''
    if github_token:
        encoded_auth = base64.b64encode(('x-access-token:' + github_token).encode()).decode()
        # Supply authorization only to GitHub via a child-process environment, never the URL/config file.
        git_env['GIT_CONFIG_COUNT'] = '1'
        git_env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
        git_env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + encoded_auth
    def run_git(arguments):
        result = subprocess.run(['git', *arguments], env=git_env, capture_output=True, text=True)
        if result.returncode:
            details = result.stderr
            for secret in (github_token, encoded_auth):
                if secret:
                    details = details.replace(secret, '[redacted]')
            if 'Could not resolve host' in details:
                raise RuntimeError('GitHub DNS lookup failed. Enable Kaggle Internet or use SOURCE_DIR.\n' + details) from None
            raise RuntimeError(
                'Git failed. For a private repository, enable GITHUB_TOKEN with read access to this repository.\n'
                + details
            ) from None
        return result.stdout.strip()
    try:
        if (PROJECT_ROOT / '.git').is_dir():
            if run_git(['-C', str(PROJECT_ROOT), 'remote', 'get-url', 'origin']) != REPO_URL:
                raise RuntimeError('Existing checkout uses another repository. Start a fresh session.')
            run_git(['-C', str(PROJECT_ROOT), 'pull', '--ff-only'])
        elif PROJECT_ROOT.exists():
            raise RuntimeError('Incomplete source folder exists. Restart the Kaggle session before retrying the clone.')
        else:
            run_git(['clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)])
    finally:
        # The token is not retained in a Git URL, command argument, or authorization config on disk.
        git_env.clear()
        github_token = encoded_auth = ''
if not (PROJECT_ROOT / MARKER).is_file():
    raise RuntimeError('The repository does not contain the expected Step 05 module.')
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
# Create .env in the private /tmp checkout, outside Kaggle saved outputs.
ENV_FILE = PROJECT_ROOT / '.env'
def dotenv_value(value):
    return "'" + value.replace('\\', '\\\\').replace("'", "\\'") + "'"
with ENV_FILE.open('w', encoding='utf-8') as handle:
    os.chmod(ENV_FILE, 0o600)
    for name in sorted(loaded_secret_names):
        if name in ENV_VARIABLE_NAMES:
            handle.write(name + '=' + dotenv_value(os.environ[name]) + '\n')
print('Private environment file created; values are hidden.')


In [ ]:
# Cell 4 — Install the checked training stack without replacing Kaggle's Torch/CUDA packages.
import subprocess
import sys
from collections import deque
from importlib.metadata import version, PackageNotFoundError

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this GPU dependency installation in Kaggle, not on the local computer.')
if any(name in sys.modules for name in ('torch', 'unsloth', 'transformers', 'peft', 'trl')):
    raise RuntimeError('Restart the Kaggle session before installing packages into an already imported training stack.')
preserved_versions = {}
for name in ('torch', 'torchvision', 'torchaudio', 'triton'):
    try:
        preserved_versions[name] = version(name)
    except PackageNotFoundError:
        if name == 'torch':
            raise RuntimeError('Select a Kaggle GPU image with CUDA Torch preinstalled.') from None
preserved_pins = [f'{name}=={release}' for name, release in preserved_versions.items()]
# The resolver must honor these pins; never use --no-deps or force a CUDA stack upgrade.
check_dns('pypi.org')

# Published Unsloth 2026.10.3 requires TRL <=1.13.0.
# TRL 1.13.0 requires Datasets >=4.7.0; Unsloth requires Datasets <5.
KAGGLE_PACKAGES = [
    'unsloth==2026.10.3',
    'unsloth_zoo==2026.10.6',
    'transformers==5.17.0',
    'trl==1.13.0',
    'datasets==4.7.0',
    'accelerate==1.15.0',
    'peft==0.21.2',
    'bitsandbytes==0.50.2',
    'wandb==0.30.0',
    'python-dotenv>=1.0.0',
    'pydantic>=2.7.0',
    'tqdm>=4.66.0',
    'matplotlib>=3.8.0',
    'typesafe-sdk>=0.7.0',
]
# Stream pip output through the notebook and include its final lines in any failure.
# CalledProcessError alone only reports an exit status; it does not explain the cause.
print('Python:', sys.version.split()[0], '| executable:', sys.executable, flush=True)
print('Preserving installed CUDA stack:', preserved_versions, flush=True)
pip_command = [
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '--progress-bar', 'off',
    '--upgrade-strategy', 'only-if-needed', *preserved_pins, *KAGGLE_PACKAGES,
]
pip_tail = deque(maxlen=80)
with subprocess.Popen(
    pip_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
        pip_tail.append(line)
    return_code = process.wait()
if return_code:
    raise RuntimeError(
        f'Kaggle dependency installation failed (pip exit code {return_code}). '
        'Do not run the model/trainer cells. Share the ERROR lines below so the '
        'failing package or dependency conflict can be identified.\n\n'
        + ''.join(pip_tail)
    )
for name, previous in preserved_versions.items():
    if version(name) != previous:
        raise RuntimeError(f'{name} changed unexpectedly. Start a fresh Kaggle GPU session.')
print('Dependency installation completed; existing CUDA package versions were preserved.', flush=True)
# If resolution fails, keep the printed ERROR lines; do not layer arbitrary Torch upgrades.
# Do not install root requirements.txt afterward: its training pins differ from this profile.
# Installation guide: https://unsloth.ai/docs/get-started/install/pip-install


In [ ]:
# Cell 5 — Import Unsloth first, check one visible GPU, then authenticate without prompts.
import os
import sys
import json
import inspect
from importlib import import_module, reload
from importlib.metadata import version

if 'unsloth' not in sys.modules and any(name in sys.modules for name in ('transformers', 'peft', 'trl')):
    raise RuntimeError('An older training stack is already imported. Start a fresh runtime and run from the top.')
from unsloth import FastLanguageModel
import torch
from trl import SFTConfig, SFTTrainer
from dotenv import load_dotenv
load_dotenv(ENV_FILE, override=False)
from huggingface_hub import login
check_dns('huggingface.co')
login(token=hf_token, add_to_git_credential=False)
if torch.cuda.device_count() != 1:
    raise RuntimeError('Expected one visible GPU. Restart the session and run from Cell 1.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
fine_tuning = reload(import_module('src.steps.05_fine_tuning.fine_tuning'))
required_helpers = ('apply_lora', 'render_training_prompt', 'build_hub_checkpoint_callback', 'download_checkpoint_from_hub')
missing = [name for name in required_helpers if not callable(getattr(fine_tuning, name, None))]
if missing:
    raise RuntimeError(f'Update the checkout: Step 05 is missing {missing}.')
report_results = import_module('src.steps.04_evaluation').report_results
runtime_versions = {}
for package in ('torch', 'unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'accelerate', 'datasets', 'bitsandbytes', 'wandb', 'tokenizers', 'pydantic', 'huggingface_hub', 'triton'):
    installed = version(package)
    runtime_versions[package] = installed
    loaded = getattr(sys.modules.get(package), '__version__', installed)
    if str(loaded) != installed:
        raise RuntimeError(f'{package} changed after import. Start a fresh runtime.')
    print(f'{package}: {installed}')
print('Trainer signature:', inspect.signature(SFTTrainer.__init__))
fields = getattr(SFTConfig, '__dataclass_fields__', {})
if not {'max_length', 'completion_only_loss'} <= set(fields):
    raise RuntimeError('Unexpected TRL API. Rerun the notebook installation cell in a fresh runtime.')
if 'max_steps' not in inspect.signature(fine_tuning.build_training_arguments).parameters:
    raise RuntimeError('Update the GitHub checkout or attached source to the Kaggle Step 05 version.')
if 'step_revisions' not in inspect.signature(fine_tuning.build_hub_checkpoint_callback).parameters:
    raise RuntimeError('Update the source checkout to support step-numbered checkpoint revisions.')
print('Fine-tuning implementation:', fine_tuning.__file__)

In [ ]:
# Cell 6 — Noninteractive feature switches and destinations; edit these values for your account.
USE_WANDB = True  # Keep experiment tracking enabled.
HUB_CHECKPOINTS = True
PUSH_TO_HUB = True
HF_PRIVATE = True  # New model repositories are private unless you change this.
RUN_BASELINE = False
RUN_POST_EVAL = True  # Evaluate held-out examples and generate results before publication.
HF_MODEL_REPO = 'md-zain/resume-parser'  # [HF_USERNAME/REPO_NAME]
HF_DATASET_REPO = 'md-zain/resume-parser-dataset'  # [HF_DATASET_REPO]
DATASET_DIR = None  # [DATASET_PATH]: optional '/kaggle/input/YOUR-DATASET/curated'
# Local data must provide train.jsonl, validation.jsonl, test.jsonl with resume_text/target_json/resume_id/category.
if USE_WANDB and not os.getenv('WANDB_API_KEY'):
    raise RuntimeError('Enable WANDB_API_KEY in Kaggle Add-ons → Secrets for this notebook before training.')
print('W&B tracking:', 'enabled' if USE_WANDB else 'disabled')
print('HF model repo:', HF_MODEL_REPO)
print('HF dataset repo:', HF_DATASET_REPO)


In [ ]:
# Cell 7 — config
# Quick validation uses a small sample and cannot establish global extraction quality.
MODEL_ID = fine_tuning.DEFAULT_MODEL_ID
LOAD_IN_4BIT = True
EVAL_SAMPLE = None  # None evaluates every retained validation example (41 in the observed run).
MAX_NEW_TOKENS = 1024  # Also capped to the selected context window minus prompt length.
WANDB_ENTITY = os.getenv('WANDB_ENTITY')
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'resume-parser')
print('Base model:', MODEL_ID)
print('Dataset source:', DATASET_DIR or HF_DATASET_REPO)

### Training limit and automatic recovery

Full training uses at most 100 optimizer steps, with a 150-minute session budget that includes setup time. The callback estimates speed after five updates and requests an earlier stop/save if needed. Uploads and an individual slow step can extend elapsed time; Kaggle can still interrupt a session. Verify remaining quota in Kaggle before a background run.

The latest **complete** checkpoint under `/kaggle/working/outputs` is resumed automatically, using its original manifest/run ID. In a fresh session, set `RESUME_HUB_REVISION` to a confirmed checkpoint SHA or experiment branch, or set `RESUME_CHECKPOINT_DIR` to a checkpoint/outputs folder attached under `/kaggle/input`. Same model, data, hyperparameters, prompt, precision, and package versions are required. Unsloth-only checkpoints are compatible with this profile; older training backends need a new experiment.

For a new experiment, use a fresh output directory/session; existing incomplete or mismatched runs are not overwritten.


In [ ]:
# Cell 8 — Configure a bounded run and recover complete checkpoints without interactive input.
from datetime import datetime, timezone
from uuid import uuid4

HYPERPARAMETERS = {
    'learning_rate': 2e-4, 'epochs': 2,
    'max_steps': 100,  # Positive max_steps overrides epochs.
    'per_device_batch_size': 1, 'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 16, 'max_seq_length': 1024,
    'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0,
    'target_modules': list(fine_tuning.DEFAULT_TARGET_MODULES),
    'optim': 'paged_adamw_8bit', 'evaluate_during_training': False,
    'warmup_ratio': 0.03, 'lr_scheduler_type': 'cosine',
    'save_steps': 50, 'save_total_limit': 2, 'eval_steps': 50,
    'max_train_samples': 2000, 'seed': 42,
}
SESSION_BUDGET_MINUTES = 150
TRAINING_DEADLINE = SESSION_STARTED_AT + SESSION_BUDGET_MINUTES * 60
OVERLENGTH_POLICY = 'drop'
PROMPT_STYLE = 'compact'
MIN_TRAIN_EXAMPLES = 64
MIN_VALIDATION_EXAMPLES = 8
RUN_TAG = 'run'  # Short checkpoint branch: run-YYYYMMDDTHHMMSSZ-<unique id>.
RUN_ID = None  # Optional identity check when recovering; otherwise read from the checkpoint manifest.
RESUME_HUB_REVISION = None  # Confirmed full checkpoint SHA or original experiment branch.
RESUME_CHECKPOINT_DIR = None  # Optional '/kaggle/input/YOUR-PREVIOUS-OUTPUT/outputs'.
OUTPUT_DIR = Path('/kaggle/working/outputs')
ADAPTER_DIR = Path('/kaggle/working/lora_adapter')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def latest_complete_checkpoint(root):
    root = Path(root)
    candidates = [root] if root.name.startswith('checkpoint-') else [
        path for path in root.glob('checkpoint-*') if path.is_dir() and path.name[11:].isdigit()
    ]
    candidates.sort(key=lambda path: int(path.name[11:]) if path.name[11:].isdigit() else -1, reverse=True)
    for candidate in candidates:
        try:
            # Reuse the module's full-state validation, including adapter and RNG files.
            step = fine_tuning._checkpoint_step(candidate)
            if candidate.name != f'checkpoint-{step}':
                raise ValueError('Directory and Trainer step disagree.')
            return candidate
        except (ValueError, KeyError, OSError):
            print('Skipping incomplete checkpoint:', candidate.name)
    return None

if RESUME_HUB_REVISION and RESUME_CHECKPOINT_DIR:
    raise ValueError('Choose either Hub recovery or attached checkpoint recovery.')
local_checkpoint = latest_complete_checkpoint(OUTPUT_DIR)
if local_checkpoint and (RESUME_HUB_REVISION or RESUME_CHECKPOINT_DIR):
    raise ValueError('A local checkpoint already exists. Remove the external recovery setting to resume it.')
if RESUME_HUB_REVISION:
    local_checkpoint = fine_tuning.download_checkpoint_from_hub(
        OUTPUT_DIR, repo_id=HF_MODEL_REPO, revision=RESUME_HUB_REVISION,
    )
elif RESUME_CHECKPOINT_DIR:
    source_checkpoint = latest_complete_checkpoint(RESUME_CHECKPOINT_DIR)
    if source_checkpoint is None:
        raise ValueError('No complete checkpoint found at RESUME_CHECKPOINT_DIR.')
    source_manifest = source_checkpoint / 'run_manifest.json'
    if not source_manifest.is_file():
        source_manifest = source_checkpoint.parent / 'run_manifest.json'
    if not source_manifest.is_file():
        raise ValueError('The attached checkpoint requires its original run_manifest.json.')
    root_manifest = OUTPUT_DIR / 'run_manifest.json'
    if root_manifest.exists() and root_manifest.read_bytes() != source_manifest.read_bytes():
        raise ValueError('Output directory contains another run manifest. Use a fresh session.')
    local_checkpoint = OUTPUT_DIR / source_checkpoint.name
    if local_checkpoint.exists():
        raise ValueError('A partial local checkpoint exists at this step. Use a fresh session before recovery.')
    shutil.copytree(source_checkpoint, local_checkpoint)
    shutil.copyfile(source_manifest, root_manifest)
manifest_path = OUTPUT_DIR / 'run_manifest.json'
if local_checkpoint and not manifest_path.is_file():
    checkpoint_manifest = local_checkpoint / 'run_manifest.json'
    if not checkpoint_manifest.is_file():
        raise ValueError('Cannot resume without the original run_manifest.json.')
    shutil.copyfile(checkpoint_manifest, manifest_path)
previous = json.loads(manifest_path.read_text()) if manifest_path.is_file() else None
if local_checkpoint:
    if RUN_ID and RUN_ID != previous['run_id']:
        raise ValueError('RUN_ID does not match the checkpoint manifest.')
    recovered_step = fine_tuning._checkpoint_step(local_checkpoint)
    if recovered_step >= HYPERPARAMETERS['max_steps']:
        raise ValueError(
            f'This run already reached step {recovered_step}, its configured training limit. '
            'Use the saved adapter/checkpoint for inference, or start a new experiment in a fresh session.'
        )
    EXPERIMENT_ID = previous['run_id']
else:
    if previous or any(OUTPUT_DIR.iterdir()):
        raise ValueError('Outputs exist without a complete checkpoint. Inspect/recover them or use a fresh session.')
    EXPERIMENT_ID = RUN_ID or f"{RUN_TAG}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex[:8]}"
RESUME_FROM_CHECKPOINT = str(local_checkpoint) if local_checkpoint else None
START_STEP = fine_tuning._checkpoint_step(local_checkpoint) if local_checkpoint else 0
TRAINED_STEP = START_STEP
RUN_NAME = f"resume-parser-step-{START_STEP}-to-{HYPERPARAMETERS['max_steps']}-{EXPERIMENT_ID}"
# Trainer and post-evaluation use the same W&B run identity.
os.environ['WANDB_RUN_ID'] = EXPERIMENT_ID
os.environ['WANDB_RESUME'] = 'allow'
print('Run ID:', EXPERIMENT_ID)
print('Resuming:', RESUME_FROM_CHECKPOINT or 'new run')
print('Output dir:', OUTPUT_DIR)
print(json.dumps(HYPERPARAMETERS, indent=2))


## 1. Download, validate, and prepare the splits

The test split is downloaded only to reserve its resume identities/text. It supplies no training targets. Training excludes all held-out overlap; duplicate texts are deduplicated, and groups with conflicting targets are excluded with counts. Original dataset files are preserved.

In [ ]:
# Cell 9 — Load the project's curated JSONL files; keep raw resumes out of saved Kaggle outputs.
if DATASET_DIR:
    data_root = Path(DATASET_DIR)
    splits = {name: data_root / f'{name}.jsonl' for name in ('train', 'validation', 'test')}
    if any(not path.is_file() for path in splits.values()):
        raise FileNotFoundError('DATASET_DIR needs train.jsonl, validation.jsonl, and test.jsonl.')
else:
    splits = fine_tuning.download_curated_splits(
        splits=('train', 'validation', 'test'), repo_id=HF_DATASET_REPO,
        curated_dir=PROJECT_ROOT / '.data/curated', force=True,
    )
raw_train_records = fine_tuning.load_records(splits['train'])
raw_validation_records = fine_tuning.load_records(splits['validation'])
test_records = fine_tuning.load_records(splits['test'])
train_records, validation_records, preparation_report = fine_tuning.prepare_training_records(
    raw_train_records, raw_validation_records, test_records=test_records,
)
print(json.dumps(preparation_report, indent=2))


In [ ]:
# Cell 10 — gpu cleanup
# Inspect the actual GPU and release only unused allocations before loading weights.
import gc
subprocess.run(['nvidia-smi'], check=True)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select a T4 runtime.')
for name in ('trainer', 'model', 'tokenizer', 'checkpoint_callback', 'callback'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print('GPU:', torch.cuda.get_device_name(0))
print('Total VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Training precision: fp16=True, bf16=False')

## 2. Load a fresh 4-bit model

Run the GPU cleanup cell first. `empty_cache()` only releases unused allocations; it cannot free a model held by old variables or an exception traceback. Use a fresh runtime after OOM. The loader uses FP16 explicitly and never loads full-precision model weights.

In [ ]:
# Cell 11 — load model
# Load the prequantized model with the same context limit used by dataset filtering.
model, tokenizer = fine_tuning.load_model_and_tokenizer(
    MODEL_ID, load_in_4bit=LOAD_IN_4BIT, max_seq_length=HYPERPARAMETERS['max_seq_length'],
)
print('Loaded:', MODEL_ID, '| compute dtype: FP16 | 4-bit: True')

## 3. Apply LoRA and prepare complete examples

Unsloth attaches adapters with `use_gradient_checkpointing="unsloth"`. Dataset preparation retains conversational prompt/completion columns so TRL masks the prompt and trains only on the JSON response. Length checks include the Qwen chat template and the whole target JSON.

In [ ]:
# Cell 12 — lora
# Attach fresh Unsloth adapters exactly once.
import hashlib
from functools import partial
from collections import Counter
prompt_fn = partial(fine_tuning.render_training_prompt, style=PROMPT_STYLE)
model = fine_tuning.apply_lora(
    model, r=HYPERPARAMETERS['lora_r'], alpha=HYPERPARAMETERS['lora_alpha'],
    dropout=HYPERPARAMETERS['lora_dropout'], target_modules=HYPERPARAMETERS['target_modules'],
    seed=HYPERPARAMETERS['seed'],
)
model.print_trainable_parameters()



In [ ]:
# Cell 13 — lengths and datasets
# Measure complete sequences and build completion-only datasets.
for name, records in [('train', train_records), ('validation', validation_records)]:
    lengths = fine_tuning.token_length_report(
        records, tokenizer, sample=None, max_length=HYPERPARAMETERS['max_seq_length'], prompt_fn=prompt_fn,
    )
    print(f'{name} complete-example length report:', json.dumps(lengths, indent=2))
fit_train_records, train_lengths = fine_tuning.filter_records_by_length(
    train_records, tokenizer, max_length=HYPERPARAMETERS['max_seq_length'],
    policy=OVERLENGTH_POLICY, prompt_fn=prompt_fn, min_samples=MIN_TRAIN_EXAMPLES,
)
fit_validation_records, validation_lengths = fine_tuning.filter_records_by_length(
    validation_records, tokenizer, max_length=HYPERPARAMETERS['max_seq_length'],
    policy=OVERLENGTH_POLICY, prompt_fn=prompt_fn, min_samples=MIN_VALIDATION_EXAMPLES,
)
print('Training exclusions:', train_lengths)
print('Validation exclusions:', validation_lengths)
train_dataset, eval_dataset = fine_tuning.build_datasets(
    fit_train_records, fit_validation_records, tokenizer=tokenizer,
    max_train=HYPERPARAMETERS['max_train_samples'],
    max_seq_length=HYPERPARAMETERS['max_seq_length'],
    seed=HYPERPARAMETERS['seed'], test_records=test_records, prompt_fn=prompt_fn,
)

# Hugging Face revision for this round, keyed to the selected training dataset size.
NUM_TRAIN_RESUMES = len(train_dataset)
REVISION = EXPERIMENT_ID  # Checkpoint history stays on a concise per-run branch; final adapter goes to main.
print(train_dataset)
print(eval_dataset)
print('Training dataset rows:', NUM_TRAIN_RESUMES, '| HF revision:', REVISION)
selected_records = fine_tuning.select_training_records(
    fit_train_records, HYPERPARAMETERS['max_train_samples'], seed=HYPERPARAMETERS['seed'],
)
from collections import Counter
print('Selected category counts:', dict(Counter(r.get('category') for r in selected_records)))


## 4. Train and retain resumable checkpoints

For another experiment, rerun the configuration, fresh model-loading, dataset, and training cells. To resume, keep the original run ID and manifest settings. Checkpoints are saved every `save_steps` and committed synchronously to Hugging Face. Interruptions can lose progress since the latest confirmed commit SHA. Local retention (`save_total_limit`) does not delete older Hub commit history.

Optional baseline evaluation runs on exactly the same prepared validation records, sample seed, and output-token budget as post-training evaluation. Old charts under `evals/` may use different samples/splits and are not a direct before/after comparison.

In [ ]:
# Cell 14 — trainer config
# Configure FP16 training and record the exact inputs needed for checkpoint recovery.
training_args = fine_tuning.build_training_arguments(
    OUTPUT_DIR,
    epochs=HYPERPARAMETERS['epochs'],
    max_steps=HYPERPARAMETERS['max_steps'],
    learning_rate=HYPERPARAMETERS['learning_rate'],
    per_device_batch_size=HYPERPARAMETERS['per_device_batch_size'],
    per_device_eval_batch_size=HYPERPARAMETERS['per_device_eval_batch_size'],
    gradient_accumulation_steps=HYPERPARAMETERS['gradient_accumulation_steps'],
    max_seq_length=HYPERPARAMETERS['max_seq_length'],
    save_steps=HYPERPARAMETERS['save_steps'],
    save_total_limit=HYPERPARAMETERS['save_total_limit'],
    eval_steps=HYPERPARAMETERS['eval_steps'],
    warmup_ratio=HYPERPARAMETERS['warmup_ratio'],
    lr_scheduler_type=HYPERPARAMETERS['lr_scheduler_type'],
    run_name=RUN_NAME,
    use_wandb=USE_WANDB, bf16=False,
    optim=HYPERPARAMETERS['optim'],
    evaluate_during_training=HYPERPARAMETERS['evaluate_during_training'],
    seed=HYPERPARAMETERS['seed'],
)
manifest = {
    'backend': 'unsloth', 'prompt_style': PROMPT_STYLE,
    'run_id': EXPERIMENT_ID, 'model_id': MODEL_ID, 'hyperparameters': HYPERPARAMETERS,
    'model_repo': HF_MODEL_REPO, 'dataset_repo': HF_DATASET_REPO,
    'runtime_versions': runtime_versions,
    'training_precision': {'bf16': training_args.bf16, 'fp16': training_args.fp16},
    'prompt_sha256': hashlib.sha256(prompt_fn('STATIC_PROMPT_MARKER').encode('utf-8')).hexdigest(),
    'schema_sha256': hashlib.sha256((PROJECT_ROOT / 'src/schema/resume_output.py').read_bytes()).hexdigest(),
    'dataset_revision': os.getenv('HF_DATASET_REVISION', 'main'),
    'split_sha256': {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in splits.items()},
    'train_resume_ids': [r['resume_id'] for r in selected_records],
    'preparation': preparation_report, 'train_lengths': train_lengths, 'validation_lengths': validation_lengths,
}
manifest_path = OUTPUT_DIR / 'run_manifest.json'
if RESUME_FROM_CHECKPOINT:
    previous = json.loads(manifest_path.read_text())
    for key in ('backend', 'prompt_style', 'run_id', 'model_id', 'model_repo', 'dataset_repo', 'hyperparameters', 'split_sha256', 'train_resume_ids', 'runtime_versions', 'training_precision', 'prompt_sha256', 'schema_sha256'):
        if previous.get(key) != manifest[key]:
            raise ValueError(f'Resume configuration changed: {key}. Use a fresh output directory/session for a different experiment.')
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
checkpoint_callback = None
if HUB_CHECKPOINTS:
    checkpoint_callback = fine_tuning.build_hub_checkpoint_callback(
        repo_id=HF_MODEL_REPO, revision=REVISION, private=HF_PRIVATE, step_revisions=True,
    )
    print('Checkpoint history branch:', REVISION)
    print('Step revision format:', f'checkpoint-step-<actual-step>-{REVISION}')
    print('Recover by full SHA at:', f'https://huggingface.co/{HF_MODEL_REPO}/commits/{REVISION}')

# Stop/save at an optimizer-step boundary; do not promise a hard limit on an individual step/upload.
from transformers import TrainerCallback
class TrainingBudgetCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.monotonic()
        self.initial_step = state.global_step
        self.planned_stop = args.max_steps
        if self.started >= TRAINING_DEADLINE:
            raise RuntimeError('The setup used the session budget. Start a fresh session to train safely.')
        return control

    def on_step_end(self, args, state, control, **kwargs):
        now = time.monotonic()
        completed = state.global_step - self.initial_step
        if completed == 5:
            seconds_per_step = (now - self.started) / completed
            remaining = max(0, TRAINING_DEADLINE - now)
            additional = int(remaining / max(seconds_per_step * 1.3, 1))
            self.planned_stop = min(args.max_steps, state.global_step + additional)
            print(f'Observed {seconds_per_step:.1f}s/update; planned stop at step {self.planned_stop}, subject to time budget.')
        if now >= TRAINING_DEADLINE or state.global_step >= self.planned_stop:
            control.should_training_stop = True
            control.should_save = True
            print('Step/session limit reached; saving full training state before export.')
        return control


def run_configured_training():
    global RUN_NAME, TRAINED_STEP
    if torch.cuda.device_count() != 1:
        raise RuntimeError('Refusing training with more than one visible GPU.')
    if USE_WANDB:
        import wandb
        if wandb.run is not None:
            wandb.finish()
        fine_tuning.configure_wandb_env(entity=WANDB_ENTITY, project=WANDB_PROJECT, run_name=RUN_NAME)
        wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, id=EXPERIMENT_ID,
            name=RUN_NAME, resume='allow', job_type='train',
            config={'model_id': MODEL_ID, **HYPERPARAMETERS},
        )
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    callbacks = [TrainingBudgetCallback()]
    if checkpoint_callback is not None:
        callbacks.append(checkpoint_callback)
    training_succeeded = False
    try:
        result = fine_tuning.train(
            model, tokenizer, train_dataset, eval_dataset, training_args=training_args,
            resume_from_checkpoint=RESUME_FROM_CHECKPOINT, callbacks=callbacks,
        )
        training_succeeded = True
        TRAINED_STEP = result.state.global_step
        RUN_NAME = f'resume-parser-step-{TRAINED_STEP}-{EXPERIMENT_ID}'
        if USE_WANDB and wandb.run is not None:
            wandb.run.name = RUN_NAME
            wandb.run.summary['training/final_step'] = TRAINED_STEP
        print('Completed run name:', RUN_NAME)
        return result
    finally:
        if USE_WANDB:
            wandb.finish(exit_code=0 if training_succeeded else 1)


In [ ]:
# Cell 15 — train
if RUN_BASELINE and not RESUME_FROM_CHECKPOINT:
    baseline_report = fine_tuning.evaluate_records(
        model, tokenizer, fit_validation_records, model_name=f'baseline:{MODEL_ID}',
        limit=None if EVAL_SAMPLE is None else min(EVAL_SAMPLE, len(fit_validation_records)), max_new_tokens=MAX_NEW_TOKENS,
        max_seq_length=HYPERPARAMETERS['max_seq_length'], prompt_fn=prompt_fn,
        seed=HYPERPARAMETERS['seed'],
        output_dir=OUTPUT_DIR / 'evaluation-before',
    )
    print('Baseline:', json.dumps(baseline_report.metrics, indent=2))

trainer = run_configured_training()
if checkpoint_callback is not None:
    print('Confirmed checkpoint commits:')
    for uploaded in checkpoint_callback.uploads:
        print(uploaded.get('step_revision', uploaded['revision']), uploaded['commit_sha'], uploaded['commit_url'])

In [ ]:
# Cell 16 — peak memory
# Peak allocator memory includes model weights and training tensors; nvidia-smi also includes CUDA overhead.
print(f'Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB')
print(f'Peak reserved GPU memory: {torch.cuda.max_memory_reserved() / 2**30:.2f} GiB')

In [ ]:
# Cell 17 — save adapter
# Save the adapter before inference/evaluation so a later generation error cannot prevent export.
adapter_dir = fine_tuning.save_adapter(trainer, tokenizer, ADAPTER_DIR)
print('Saved adapter to:', adapter_dir)


## 5. Post-training evaluation on held-out validation data

Post-evaluation is enabled by default. `EVAL_SAMPLE=None` evaluates every retained validation example: **41 in the observed run**, after duplicate/leakage and 1,024-token checks. The untouched test split remains reserved. These short-context validation results are not a benchmark for excluded long resumes.

The tables show per-resume status, JSON/schema validity, programmatic, semantic, overall and identity scores, and aggregate field scores. Semantic comparisons require `TYPESAFE_API_KEY`; missing semantic scores remain unscored with visible status counts. This generation evaluation does not measure validation loss or prove the absence of overfitting.

The evaluation-model cell reuses the trained model or reloads the saved adapter/local checkpoint if the model is unavailable. Explicit Hub recovery is also available via `EVAL_ADAPTER_REVISION`. No training is required to reload the adapter. Final main publication follows evaluation. After a complete session reset, restore the source/dependencies and validation-data preparation first.


In [ ]:
# Cell 18 — evaluation model
# Reuse the trained model or reload a saved adapter for evaluation.
from pathlib import Path
from unsloth import FastLanguageModel
import torch

EVAL_ADAPTER_DIR = None  # Optional local adapter/checkpoint folder, including a surviving checkpoint-100.
EVAL_ADAPTER_REVISION = None  # Optional Hub branch/SHA; overrides the current model and local adapter.
EVAL_ADAPTER_REPO = HF_MODEL_REPO
ADAPTER_FILES = [
    'adapter_config.json', 'adapter_model*.safetensors', 'adapter_model*.bin',
    'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
    'added_tokens.json', 'vocab.json', 'merges.txt', 'chat_template.jinja', 'chat_templates/*.jinja',
]

def has_saved_adapter(folder):
    return (folder / 'adapter_config.json').is_file() and bool(
        list(folder.glob('adapter_model*.safetensors')) or list(folder.glob('adapter_model*.bin'))
    )

def resolve_saved_adapter():
    if EVAL_ADAPTER_DIR and EVAL_ADAPTER_REVISION:
        raise ValueError('Choose a local adapter folder or a Hub revision, not both.')
    if EVAL_ADAPTER_DIR:
        candidate = Path(EVAL_ADAPTER_DIR)
        if not has_saved_adapter(candidate):
            raise FileNotFoundError('EVAL_ADAPTER_DIR needs adapter_config.json and adapter weights.')
        return candidate
    if not EVAL_ADAPTER_REVISION:
        candidate = Path(ADAPTER_DIR)
        if has_saved_adapter(candidate):
            return candidate
        # Local checkpoints can survive even if the remote repository has been deleted.
        candidates = [p for p in Path(OUTPUT_DIR).glob('checkpoint-*') if p.name[11:].isdigit()]
        for candidate in sorted(candidates, key=lambda p: int(p.name[11:]), reverse=True):
            try:
                fine_tuning._checkpoint_step(candidate)
            except (ValueError, KeyError, OSError):
                continue
            if has_saved_adapter(candidate):
                return candidate
    revision = EVAL_ADAPTER_REVISION or globals().get('REVISION')
    if not revision:
        raise RuntimeError('No local trained adapter/checkpoint is available. Select an existing uploaded run revision.')
    from huggingface_hub import snapshot_download
    snapshot = Path(snapshot_download(
        repo_id=EVAL_ADAPTER_REPO, revision=revision, token=os.getenv('HF_TOKEN'),
        allow_patterns=ADAPTER_FILES + ['checkpoint/' + pattern for pattern in ADAPTER_FILES],
    ))
    candidate = snapshot if has_saved_adapter(snapshot) else snapshot / 'checkpoint'
    if not has_saved_adapter(candidate):
        raise RuntimeError('The selected Hub revision contains no trained adapter. Select the uploaded run branch/SHA.')
    print('Evaluation adapter snapshot SHA:', snapshot.name)
    return candidate


def get_evaluation_model():
    explicit_source = bool(EVAL_ADAPTER_DIR or EVAL_ADAPTER_REVISION)
    active_trainer = globals().get('trainer')
    trained_model = getattr(active_trainer, 'model', None)
    trained_tokenizer = globals().get('tokenizer')
    if not explicit_source and trained_model is not None and trained_tokenizer is not None:
        if not getattr(trained_model, 'peft_config', None):
            raise RuntimeError('Evaluation requires the trained LoRA adapter, not a bare base model.')
        return trained_model, trained_tokenizer
    source_key = (str(EVAL_ADAPTER_DIR or ADAPTER_DIR), EVAL_ADAPTER_REPO, EVAL_ADAPTER_REVISION or REVISION)
    if (globals().get('_evaluation_source_key') == source_key
            and globals().get('evaluation_model') is not None
            and globals().get('evaluation_tokenizer') is not None):
        return evaluation_model, evaluation_tokenizer
    adapter_path = resolve_saved_adapter()
    # Release all notebook references before loading another copy on the single T4.
    active_trainer = trained_model = trained_tokenizer = None
    for name in ('trainer', 'model', 'tokenizer', 'callback', 'evaluation_model', 'evaluation_tokenizer'):
        globals().pop(name, None)
    fine_tuning.free_memory()
    recovered_model, recovered_tokenizer = fine_tuning.load_model_and_tokenizer(
        str(adapter_path), load_in_4bit=True, max_seq_length=HYPERPARAMETERS['max_seq_length'],
    )
    if not getattr(recovered_model, 'peft_config', None):
        raise RuntimeError('The trained adapter was not attached. Refusing to evaluate an untuned base model.')
    FastLanguageModel.for_inference(recovered_model)
    globals()['evaluation_model'] = recovered_model
    globals()['evaluation_tokenizer'] = recovered_tokenizer
    globals()['_evaluation_source_key'] = source_key
    print('Reloaded trained adapter for evaluation:', adapter_path)
    return recovered_model, recovered_tokenizer


In [ ]:
# Cell 19 — quick inference
# Use synthetic text for a quick generation check without printing private resumes.
evaluation_model, evaluation_tokenizer = get_evaluation_model()
quick_item = fine_tuning.EvaluationItem(
    resume_id='synthetic-smoke-check',
    resume_text='Alex Example is a Python developer. Worked at Example Company from 2021 to 2024. Skills: Python, SQL.',
    target_json={},
)
callback = fine_tuning.make_generation_callback(
    evaluation_model, evaluation_tokenizer, model_name='unsloth-qlora', prompt_fn=prompt_fn,
    max_seq_length=HYPERPARAMETERS['max_seq_length'], max_new_tokens=MAX_NEW_TOKENS,
)
quick_output = callback(quick_item)
print(quick_output.raw_output)
try:
    quick_json = json.loads(quick_output.raw_output)
    from src.schema.resume_output import ResumeOutput
    ResumeOutput.model_validate(quick_json)
    print('Synthetic output is valid JSON matching ResumeOutput.')
except (ValueError, TypeError):
    print('Synthetic output is incomplete or invalid. Inspect the output and training length coverage.')

In [ ]:
# Cell 20 — evaluate
# Evaluate every retained held-out validation resume, then save/display anonymous score tables.
if RUN_POST_EVAL:
    if 'fit_validation_records' not in globals() or not fit_validation_records:
        raise RuntimeError('Prepare the held-out validation records before evaluation. Do not rerun training.')
    if EVAL_SAMPLE is not None and (isinstance(EVAL_SAMPLE, bool) or not isinstance(EVAL_SAMPLE, int) or EVAL_SAMPLE < 1):
        raise ValueError('EVAL_SAMPLE must be None (all retained validation records) or a positive integer.')
    evaluation_model, evaluation_tokenizer = get_evaluation_model()
    evaluated_adapter_source = str(resolve_saved_adapter().resolve())
    evaluation_limit = None if EVAL_SAMPLE is None else min(EVAL_SAMPLE, len(fit_validation_records))
    print('Post-evaluation resumes:', evaluation_limit or len(fit_validation_records))
    if not os.getenv('TYPESAFE_API_KEY'):
        print('TYPESAFE_API_KEY is unset: programmatic checks still run; unavailable semantic scores stay unscored.')
    report = fine_tuning.evaluate_records(
        evaluation_model, evaluation_tokenizer, fit_validation_records,
        model_name=f'finetuned:{MODEL_ID}', limit=evaluation_limit,
        max_new_tokens=MAX_NEW_TOKENS, max_seq_length=HYPERPARAMETERS['max_seq_length'],
        prompt_fn=prompt_fn, seed=HYPERPARAMETERS['seed'],
        # Detailed results contain resume text/targets; keep them outside saved Kaggle outputs.
        output_dir=PROJECT_ROOT / '.data/post-evaluation' / EXPERIMENT_ID,
    )
    import pandas as pd
    from IPython.display import display
    from collections import defaultdict
    metrics = {key: value for key, value in report.metrics.items() if key != 'selected_resume_ids'}
    ordered_results = sorted(report.results, key=lambda result: result.item.resume_id)
    evaluation_table = pd.DataFrame([
        {
            'case': f'validation-{index:03d}', 'category': result.item.category,
            'status': result.status, 'json_valid': result.json_valid, 'schema_valid': result.schema_valid,
            'programmatic_score': result.programmatic_score, 'semantic_score': result.semantic_score,
            'overall_score': result.overall_score, 'identity_score': result.identity_score,
            'duration_seconds': result.duration_seconds,
        }
        for index, result in enumerate(ordered_results, 1)
    ])
    total = len(ordered_results)
    summary_rows = [
        {'metric': 'Resumes evaluated', 'value': total},
        {'metric': 'JSON valid count', 'value': metrics['json_valid_count']},
        {'metric': 'JSON valid rate', 'value': metrics['json_valid_count'] / total if total else None},
        {'metric': 'Schema valid count', 'value': metrics['schema_valid_count']},
        {'metric': 'Schema valid rate', 'value': metrics['schema_valid_count'] / total if total else None},
        {'metric': 'Fully scored resumes', 'value': metrics['scored_count']},
        {'metric': 'Mean programmatic field score', 'value': metrics['mean_programmatic_score']},
        {'metric': 'Mean semantic field score', 'value': metrics['mean_semantic_score']},
        {'metric': 'Mean overall score among scored resumes', 'value': metrics['mean_overall_score']},
        {'metric': 'Mean identity score', 'value': metrics['mean_identity_score']},
    ]
    evaluation_summary_table = pd.DataFrame(summary_rows)
    field_values = defaultdict(list)
    for result in ordered_results:
        for field in result.score_breakdown:
            if field.get('value') is not None:
                field_values[(field['group'], field['path'])].append(float(field['value']))
    evaluation_field_table = pd.DataFrame([
        {'group': group, 'field': field, 'mean_score': sum(values) / len(values), 'scored_comparisons': len(values)}
        for (group, field), values in sorted(field_values.items())
    ], columns=['group', 'field', 'mean_score', 'scored_comparisons'])
    display(evaluation_summary_table, evaluation_table, evaluation_field_table)
    print('Status counts:', metrics['status_counts'])
    print('Scores/rates use 0–1. Missing scores are unscored; they are not perfect scores or silently replaced with zero.')
    report_dir = OUTPUT_DIR / 'evaluation-after'
    report_dir.mkdir(parents=True, exist_ok=True)
    evaluation_summary_table.to_csv(report_dir / 'summary.csv', index=False)
    evaluation_table.to_csv(report_dir / 'per_resume.csv', index=False)
    evaluation_field_table.to_csv(report_dir / 'per_field.csv', index=False)
    evaluation_summary = {
        'run_id': EXPERIMENT_ID, 'model_id': MODEL_ID, 'model_repo': HF_MODEL_REPO,
        'checkpoint_revision': REVISION, 'adapter_source': evaluated_adapter_source,
        'validation_scope': 'retained examples fitting the configured context',
        'max_seq_length': HYPERPARAMETERS['max_seq_length'], 'max_new_tokens': MAX_NEW_TOKENS,
        'semantic_key_available': bool(os.getenv('TYPESAFE_API_KEY')), 'metrics': metrics,
    }
    (report_dir / 'metrics.json').write_text(json.dumps(evaluation_summary, indent=2) + '\n', encoding='utf-8')
    print('Saved anonymous evaluation tables:', report_dir)
    if USE_WANDB:
        import wandb
        evaluation_run = wandb.run
        owns_evaluation_run = evaluation_run is None
        if owns_evaluation_run:
            evaluation_run = wandb.init(
                project=WANDB_PROJECT, entity=WANDB_ENTITY, id=EXPERIMENT_ID,
                name=RUN_NAME, resume='allow', job_type='post-evaluation',
            )
        try:
            evaluation_run.log({
                **{f'post_eval/{key}': value for key, value in metrics.items() if isinstance(value, (int, float))},
                'post_eval/per_resume': wandb.Table(dataframe=evaluation_table),
                'post_eval/summary': wandb.Table(dataframe=evaluation_summary_table),
                'post_eval/per_field': wandb.Table(dataframe=evaluation_field_table),
            })
            evaluation_run.summary['post_eval/status_counts'] = metrics['status_counts']
            evaluation_run.summary['post_eval/validation_scope'] = evaluation_summary['validation_scope']
        finally:
            if owns_evaluation_run:
                evaluation_run.finish()
else:
    print('Post-evaluation skipped by explicit configuration. Set RUN_POST_EVAL=True to produce validation tables.')


### Publish the final evaluated adapter to main

Checkpoint history stays on the unique `run-<UTC timestamp>-<id>` branch. Each checkpoint SHA also has a readable branch such as `checkpoint-step-50-run-<UTC timestamp>-<id>` or `checkpoint-step-100-run-<UTC timestamp>-<id>`. After post-evaluation and W&B result logging, the final inference adapter/tokenizer and aggregate evaluation summary are uploaded to **main**. This cell uses saved files, so it also works after the Trainer has been removed. The HF token creates the new repository automatically when this cell or a checkpoint callback runs.


In [ ]:
# Cell 21 — push
# Publish the evaluated adapter to main from saved files; a Trainer object is not required.
if PUSH_TO_HUB:
    if RUN_POST_EVAL and 'evaluation_summary' not in globals():
        raise RuntimeError('Run post-evaluation before publishing the final adapter to main.')
    from huggingface_hub import HfApi
    from tempfile import TemporaryDirectory
    import shutil
    source_adapter = resolve_saved_adapter()
    if RUN_POST_EVAL and (
        evaluation_summary['run_id'] != EXPERIMENT_ID
        or evaluation_summary['adapter_source'] != str(source_adapter.resolve())
    ):
        raise RuntimeError('The selected adapter/run changed after evaluation. Evaluate this adapter before publication.')
    api = HfApi(token=os.getenv('HF_TOKEN'))
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type='model', private=HF_PRIVATE, exist_ok=True)
    # Stage only inference files and anonymous aggregate metadata, never optimizer state, secrets, or resumes.
    with TemporaryDirectory(dir=PRIVATE_ROOT, prefix='release-') as staging:
        release_dir = Path(staging)
        for pattern in ADAPTER_FILES:
            for source_file in source_adapter.glob(pattern):
                if source_file.is_file():
                    target = release_dir / source_file.relative_to(source_adapter)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source_file, target)
        if not has_saved_adapter(release_dir):
            raise RuntimeError('The release is missing its trained LoRA adapter.')
        if RUN_POST_EVAL:
            (release_dir / 'evaluation_summary.json').write_text(
                json.dumps(evaluation_summary, indent=2) + '\n', encoding='utf-8',
            )
        release_metadata = {
            'run_id': EXPERIMENT_ID, 'base_model': MODEL_ID,
            'checkpoint_revision': REVISION, 'training_hyperparameters': HYPERPARAMETERS,
            'post_evaluation_enabled': RUN_POST_EVAL,
        }
        (release_dir / 'release_metadata.json').write_text(json.dumps(release_metadata, indent=2) + '\n', encoding='utf-8')
        (release_dir / 'README.md').write_text(
            '---\nbase_model: ' + MODEL_ID + '\nlibrary_name: peft\ntags:\n- qlora\n- resume-parsing\n---\n'
            '# Resume Parser\n\n'
            'LoRA adapter for extracting structured resume JSON using the project ResumeOutput layout.\n\n'
            'Load with its 4-bit base model and use the compact extraction prompt from Step 05. '
            'The published files contain adapter weights and tokenizer files, not a merged full model.\n\n'
            'See evaluation_summary.json for measured validation results when enabled. '
            'Those results cover only retained validation examples fitting the configured context; '
            'they do not establish accuracy on all resume lengths or categories.\n',
            encoding='utf-8',
        )
        commit = api.upload_folder(
            repo_id=HF_MODEL_REPO, repo_type='model', folder_path=str(release_dir), revision='main',
            allow_patterns=ADAPTER_FILES + ['README.md', 'evaluation_summary.json', 'release_metadata.json'],
            commit_message=f'Publish evaluated resume parser adapter ({EXPERIMENT_ID})', run_as_future=False,
        )
    print('Published model:', f'https://huggingface.co/{HF_MODEL_REPO}/tree/main')
    print('Release commit SHA:', commit.oid)
    print('Release commit:', commit.commit_url)
else:
    print('Adapter remains saved locally; publication disabled.')


## If Kaggle still runs out of memory

Use a fresh runtime after an OOM, then change one setting at a time and use a fresh output directory/session:

1. Reduce `max_seq_length` from 1024 to 512 **only if the complete-example report shows enough usable data**. Raising it to 2048/4096 admits longer resumes but increases memory use.
2. Reduce `lora_r` from 16 to 8, then 4; set `lora_alpha` to the same value for this profile.
3. Reduce `target_modules` to `['q_proj', 'v_proj']`.
4. Switch `MODEL_ID` to `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` and use a distinct run ID/repository revision.

Keep batch size 1, FP16, 4-bit loading, Unsloth checkpointing, and no periodic evaluation. Gradient accumulation changes the effective batch size; lowering it generally does not fix the memory needed by one sequence. `empty_cache()` cannot make live tensors disappear.

A small context budget can discard most long resumes or stop preparation entirely. Do not silently truncate target JSON to force training to proceed. Review retained counts/category coverage before interpreting model scores. GPU fit, installation, and model quality must be verified in Kaggle; this notebook was edited with static checks only.

### Values to customize before Save & Run All

- **Source cell:** `REPO_URL` (`[REPO_URL]`); the current project URL is filled in. For private GitHub access, enable `GITHUB_TOKEN` with read access. If GitHub cannot be reached, attach source as a private Kaggle input and set `SOURCE_DIR` to the directory containing `src/`.
- **Secrets cell:** `ENV_VARIABLE_NAMES` (`[LIST OF ENV VARIABLE NAMES]`). Enable `HF_TOKEN` with write access. W&B tracking is enabled by default and requires `WANDB_API_KEY`; `WANDB_PROJECT` and `WANDB_ENTITY` select its destination. Typesafe credentials are optional unless semantic evaluation is enabled. Values are never printed and the generated `.env` stays under `/tmp`.
- **Feature-switch cell:** `HF_MODEL_REPO` (`[HF_USERNAME/REPO_NAME]`), `HF_DATASET_REPO`, and optional `DATASET_DIR` (`[DATASET_PATH]`). The new model repository is `md-zain/resume-parser`; the dataset repository is unchanged. Local curated JSONL files must have `resume_text`, `target_json`, `resume_id`, and `category` fields.
- **Training limits:** review `max_steps`, `SESSION_BUDGET_MINUTES`, context coverage, and remaining Kaggle quota. Defaults do not guarantee three-hour completion.
- **Recovery in a fresh session:** set `RESUME_HUB_REVISION` to a confirmed checkpoint SHA/branch, or attach prior outputs and set `RESUME_CHECKPOINT_DIR`. Existing local complete checkpoints resume automatically. Leave recovery values unset for a new run.

If cloning still reports `Could not resolve host: github.com`, enable **Internet** in Kaggle settings and retry in a fresh session. A token cannot repair DNS. An attached source folder avoids the GitHub clone, but package installation/model/dataset downloads and Hub uploads still require network access. Make sure the latest notebook **and module changes** are available in the chosen repository/source folder.

References: [Kaggle notebook settings](https://www.kaggle.com/docs/notebooks), [Kaggle Secrets implementation](https://github.com/Kaggle/docker-python/blob/main/patches/kaggle_secrets.py), [Unsloth installation](https://unsloth.ai/docs/get-started/install/pip-install). Only static checks were performed locally; installation, GPU memory, background execution, and Hub transfers must be verified on Kaggle.

- **Evaluation recovery:** `EVAL_ADAPTER_DIR` can select a local/attached trained adapter, or `EVAL_ADAPTER_REPO` and `EVAL_ADAPTER_REVISION` can select the uploaded run branch/checkpoint SHA. When unset, the current trained model or local adapter export is used. The trained model stays available for manual evaluation.

- **Post-evaluation and release:** `RUN_POST_EVAL=True` and `EVAL_SAMPLE=None` evaluate all retained validation records before the final upload to `main`. Checkpoint snapshots use `checkpoint-step-<actual step>-run-<UTC timestamp>-<unique id>`; the run branch keeps their complete commit history. W&B names include the starting/target steps while training and the actual completed step afterward, for example `resume-parser-step-100-run-<UTC timestamp>-<unique id>`. W&B uses the same experiment ID for training and post-evaluation logs.
